In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install monai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 102.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
accelerate 1.5.2 requires torch>=2.0.0, but you have torch 1.13.1 which is incompatible.
torchvision 0.21.0+cu124 requires torch==2.6.0, but you have torch 1.13.1 which is incompatible.


In [2]:
# test_pretrained.py
import os
import torch
import monai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd,
    NormalizeIntensityd, Spacingd, Orientationd, EnsureTyped, Resized
)
from monai.data import Dataset, DataLoader
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.utils import set_determinism

# Set deterministic behavior for reproducibility
set_determinism(seed=42)

# Configure paths
data_dir = "/content/drive/MyDrive/file_data"  # Update with actual path
model_path = "/content/model.pt"  # Path to pre-trained model

# Create test dataset from the CSV
df = pd.read_csv('/content/cohort(in).csv')
test_files = []

for _, row in df.iterrows():
    test_files.append({
        "image": os.path.join(data_dir, row['T2W_NIFTI']),
        "label": os.path.join(data_dir, row['Gland_NIFTI'])
    })

# Define test transforms - ADD RESIZE TRANSFORM to ensure consistent dimensions
test_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(
        keys=["image", "label"],
        pixdim=[0.5, 0.5, 0.5],
        mode=("bilinear", "nearest")
    ),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    # Add resize to ensure consistent dimensions
    Resized(
        keys=["image", "label"],
        spatial_size=(512, 512, 64),  # Match the model's expected output size
        mode=("bilinear", "nearest")
    ),
    ScaleIntensityd(keys=["image"], minv=0, maxv=1),
    NormalizeIntensityd(keys=["image"]),
    EnsureTyped(keys=["image", "label"])
])

# Create test dataset and loader
test_ds = Dataset(data=test_files, transform=test_transforms)
test_loader = DataLoader(test_ds, batch_size=1, num_workers=2)

# Determine device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pre-trained model
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,  # Model has 3 output channels
    channels=[16, 32, 64, 128, 256, 512],
    strides=[2, 2, 2, 2, 2],
    num_res_units=4,
    act="PRELU",
    norm="BATCH",
    dropout=0.15
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# Define metrics
dice_metric = DiceMetric(include_background=False, reduction="mean")

# Create directory for visualizations
os.makedirs('segmentation_results', exist_ok=True)

# Evaluation
with torch.no_grad():
    dice_scores = []

    for idx, test_data in enumerate(test_loader):
        print(f"Processing case {idx+1}/{len(test_loader)}")
        test_inputs, test_labels = (
            test_data["image"].to(device),
            test_data["label"].to(device)
        )

        print(f"Input shape: {test_inputs.shape}")
        print(f"Label shape: {test_labels.shape}")

        # Make prediction
        test_outputs = model(test_inputs)
        print(f"Output shape: {test_outputs.shape}")

        # Apply softmax to get probabilities
        test_outputs = torch.softmax(test_outputs, dim=1)

        # Convert binary label to one-hot with 3 channels to match model output
        test_labels_binary = (test_labels > 0.5).float()
        test_labels_onehot = torch.zeros_like(test_outputs)
        test_labels_onehot[:, 0, ...] = 1 - test_labels_binary.squeeze(1)  # Background
        test_labels_onehot[:, 1, ...] = test_labels_binary.squeeze(1)      # Prostate
        # Channel 2 stays zeros (assumed to be another segment not in your binary data)

        # Get the most relevant channel from the model output (likely channel 1)
        relevant_output = test_outputs[:, 1:2, ...]  # Take channel 1 (prostate gland)
        relevant_label = test_labels_binary

        # Check dice directly on binary predictions
        pred_binary = (relevant_output > 0.5).float()

        # Calculate dice manually for verification
        intersection = torch.sum(pred_binary * relevant_label)
        union = torch.sum(pred_binary) + torch.sum(relevant_label)
        manual_dice = 2.0 * intersection / (union + 1e-5)

        # Also try MONAI's dice metric
        try:
            dice_metric(y_pred=pred_binary, y=relevant_label)
            current_dice = dice_metric.aggregate().item()
            dice_metric.reset()
        except Exception as e:
            print(f"Error computing metrics with MONAI: {e}")
            current_dice = manual_dice.item()

        dice_scores.append(current_dice)
        print(f"  - Dice: {current_dice:.4f}")

        # Visualization - Create mid-axial view
        slice_idx = test_inputs.shape[4] // 2  # Middle slice in z-dimension

        # Extract middle slices
        input_slice = test_inputs[0, 0, :, :, slice_idx].cpu().numpy()
        label_slice = test_labels[0, 0, :, :, slice_idx].cpu().numpy()
        pred_slice = pred_binary[0, 0, :, :, slice_idx].cpu().numpy()

        # Create a figure with three subplots
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # Display input image
        axes[0].imshow(input_slice, cmap='gray')
        axes[0].set_title('Input T2 Image')
        axes[0].axis('off')

        # Display ground truth
        axes[1].imshow(input_slice, cmap='gray')
        axes[1].imshow(label_slice, cmap='Reds', alpha=0.3)
        axes[1].set_title('Ground Truth Segmentation')
        axes[1].axis('off')

        # Display prediction
        axes[2].imshow(input_slice, cmap='gray')
        axes[2].imshow(pred_slice, cmap='Blues', alpha=0.3)
        axes[2].set_title(f'Model Prediction (Dice: {current_dice:.4f})')
        axes[2].axis('off')

        # Save figure
        plt.tight_layout()
        plt.savefig(f'segmentation_results/case_{idx+1}_slice_{slice_idx}.png', dpi=150)
        plt.close()

        # Create mid-sagittal view
        sag_slice_idx = test_inputs.shape[2] // 2  # Middle slice in x-dimension

        # Extract middle slices (sagittal view)
        input_sag = test_inputs[0, 0, sag_slice_idx, :, :].cpu().numpy()
        label_sag = test_labels[0, 0, sag_slice_idx, :, :].cpu().numpy()
        pred_sag = pred_binary[0, 0, sag_slice_idx, :, :].cpu().numpy()

        # Create a figure with three subplots (sagittal view)
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # Display input image
        axes[0].imshow(input_sag, cmap='gray')
        axes[0].set_title('Input T2 Image (Sagittal)')
        axes[0].axis('off')

        # Display ground truth
        axes[1].imshow(input_sag, cmap='gray')
        axes[1].imshow(label_sag, cmap='Reds', alpha=0.3)
        axes[1].set_title('Ground Truth Segmentation')
        axes[1].axis('off')

        # Display prediction
        axes[2].imshow(input_sag, cmap='gray')
        axes[2].imshow(pred_sag, cmap='Blues', alpha=0.3)
        axes[2].set_title(f'Model Prediction')
        axes[2].axis('off')

        # Save figure
        plt.tight_layout()
        plt.savefig(f'segmentation_results/case_{idx+1}_sagittal_slice_{sag_slice_idx}.png', dpi=150)
        plt.close()

        # Also create 3D visualizations for selected cases (e.g., every 10th case)
        if idx % 10 == 0:
            # Create multiple slice visualizations
            for z_slice in range(0, test_inputs.shape[4], 8):  # Sample every 8 slices
                if z_slice >= test_inputs.shape[4]:
                    continue

                input_slice = test_inputs[0, 0, :, :, z_slice].cpu().numpy()
                label_slice = test_labels[0, 0, :, :, z_slice].cpu().numpy()
                pred_slice = pred_binary[0, 0, :, :, z_slice].cpu().numpy()

                # Create a figure with three subplots
                fig, axes = plt.subplots(1, 3, figsize=(18, 6))

                # Display input image
                axes[0].imshow(input_slice, cmap='gray')
                axes[0].set_title(f'Input T2 Image (Slice {z_slice})')
                axes[0].axis('off')

                # Display ground truth
                axes[1].imshow(input_slice, cmap='gray')
                axes[1].imshow(label_slice, cmap='Reds', alpha=0.3)
                axes[1].set_title('Ground Truth Segmentation')
                axes[1].axis('off')

                # Display prediction
                axes[2].imshow(input_slice, cmap='gray')
                axes[2].imshow(pred_slice, cmap='Blues', alpha=0.3)
                axes[2].set_title(f'Model Prediction')
                axes[2].axis('off')

                # Save figure
                plt.tight_layout()
                plt.savefig(f'segmentation_results/case_{idx+1}_3d_slice_{z_slice}.png', dpi=150)
                plt.close()

    # Calculate overall statistics
    if dice_scores:
        mean_dice = np.mean(dice_scores)
        std_dice = np.std(dice_scores)

        print("\nOverall Performance on D2 Dataset:")
        print(f"Dice Score: {mean_dice:.4f} ± {std_dice:.4f}")

        # Save results to CSV
        results_df = pd.DataFrame({
            'case_id': list(range(1, len(dice_scores) + 1)),
            'dice_score': dice_scores,
        })

        results_df.to_csv('pretrained_model_results.csv', index=False)
        print("Results saved to pretrained_model_results.csv")

        # Create a histogram of Dice scores
        plt.figure(figsize=(10, 6))
        plt.hist(dice_scores, bins=20, alpha=0.7, color='blue')
        plt.axvline(mean_dice, color='red', linestyle='dashed', linewidth=2, label=f'Mean Dice: {mean_dice:.4f}')
        plt.xlabel('Dice Score')
        plt.ylabel('Number of Cases')
        plt.title('Distribution of Dice Scores on D2 Dataset')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig('segmentation_results/dice_score_distribution.png', dpi=150)
        plt.close()
    else:
        print("No valid dice scores were calculated.")

Using device: cuda
Processing case 1/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.0000
Processing case 2/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.1875
Processing case 3/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.5977
Processing case 4/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.5208
Processing case 5/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.6671
Processing case 6/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1